In [62]:
import re
import ast
from langchain_ollama import OllamaLLM
import pandas as pd

model = OllamaLLM(model="mistral")

stigmatizingLanguageFound = ['noncompliant']

def group_by_second_index(data):
    result = {}

    for element in data:
        key = element[1]  # The second index (the grouping key)
        value = element[0]  # The first index (the value for the key)

        if key in result:
            result[key].append(value)  # If the key exists, append the value to the list
        else:
            result[key] = [value]  # If the key doesn't exist, create a new list with the value

    return result

def askOllama(prompt):
    result = model.invoke(input=prompt)
    return result

def cleanOllamaOutput(output):
    pattern = r"\[.*?\]"
    
    matches = re.findall(pattern, output, re.DOTALL)
    a = matches[0].replace("\n", "")
    escaped_string = re.sub(r"(?<=\w)'(?=\w)", r"\'", a)
    result = re.sub(r"\([^()]*\)", "", escaped_string)
    return ast.literal_eval(result.replace("\\n", "").replace("\\\\", "\\"))

def replaceStigmatizingLanguage(df):
    global stigmatizingLanguageFound
    clinicalNote = df.iloc[0]['Completion']
    ogClinicalNote = df.iloc[0]['Completion']
    allList = []
    print(stigmatizingLanguageFound)
    for index, i in enumerate([stigmatizingLanguageFound]):
        clinicalNote = df.iloc[index]["Completion"]
        clinicalNote = re.sub(r'^.*?\*\*History of Present Illness:\*\*', '', clinicalNote, flags=re.DOTALL)
        sentences = clinicalNote.split("**")
        sentences = [item for part in sentences for item in part.split("-")]
        for word in i:
            if len([j for j in sentences if word.lower() in j.lower()]) > 0:
                text = [j for j in sentences if word.lower() in j.lower()]
                for x in text:
                    allList.append([word, x])
    print(allList)
    newDict = {}
    grouped_list = group_by_second_index(allList)
    print(grouped_list)
    for key, value in grouped_list.items():
        replacingPrompt = "You are a professional linguist whose job is to replace stigmatizing language in clinical notes. If you see labels such as diabetic or abuser, replace these labels with person first language such as \"person who has diabetes\" or\"person with a substance abuse disorder\". If you see words like challenging or uncooperative, replace them with more respectful alternatives. Here is the sentence: " + str(key) + " And here is are the stigmatizing words you must replace: " + str(value) + ". Return to me a JSON object containing only the corrected sentence in a list"
        while True:
            try:
                newDict[key] = cleanOllamaOutput(askOllama(replacingPrompt))[0]
                break
            except:
                pass
    newClinicalNote = ogClinicalNote
    print("OG CLINICAL NOTE:")
    print(ogClinicalNote)
    for key, value in newDict.items():
        print("KEY:")
        print(key)
        if key[-2:] == "\n\n":
            value += "\n\n"
        print("VALUE:")
        print(value)
        newClinicalNote = newClinicalNote.replace(key, value)
    print("NEW CLINICAL NOTE:")
    print(newClinicalNote)
    return newClinicalNote, list(newDict.values())
df = pd.read_csv("/Users/sagewong/git/StigmatizingLanguageProject/FinalFinalAnnnotatedData.csv")
modified_text_content, correctedLanguageList = replaceStigmatizingLanguage(df)


['noncompliant']
[['noncompliant', ' Mohammad, a challenging patient with a history of uncooperative behavior, presents with concerns about his significantly short stature compared to peers. Despite recommendations for nutritional interventions, Mohammad has been noncompliant, citing his preferences for junk food. His mother reports frequent cursing at nurses during previous medical consultations, further complicating collaborative treatment efforts.\n\n'], ['noncompliant', ' Noncompliant with nutritional recommendations\n']]
{' Mohammad, a challenging patient with a history of uncooperative behavior, presents with concerns about his significantly short stature compared to peers. Despite recommendations for nutritional interventions, Mohammad has been noncompliant, citing his preferences for junk food. His mother reports frequent cursing at nurses during previous medical consultations, further complicating collaborative treatment efforts.\n\n': ['noncompliant'], ' Noncompliant with nut

In [58]:
clinicalNote = df.iloc[0]['Completion']
sentences = clinicalNote.split("**")
sentences = [item for part in sentences for item in part.split("-")]

for i in sentences:
    if "noncompliant" in i.lower():
        print(i)

 Mohammad, a challenging patient with a history of uncooperative behavior, presents with concerns about his significantly short stature compared to peers. Despite recommendations for nutritional interventions, Mohammad has been noncompliant, citing his preferences for junk food. His mother reports frequent cursing at nurses during previous medical consultations, further complicating collaborative treatment efforts.


 Noncompliant with nutritional recommendations

